# Representax TPU acceptance

Select **Runtime > Change runtime type > TPU v2**, then press **Run all**. The revision is prefilled below and the notebook executes the repository-owned toy acceptance matrix; it does not contain another training implementation.

In [ ]:
import os
import pathlib
import subprocess
import sys

REPOSITORY = "https://github.com/ckgresla/representax.git"
REVISION = "codex/colab-tpu-acceptance" # @param {type:"string"}
CHECKOUT = pathlib.Path("/content/representax")

if not CHECKOUT.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", REPOSITORY, str(CHECKOUT)], check=True)
subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "origin", REVISION], check=True)
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{CHECKOUT}[tpu]"], check=True)
REVISION = subprocess.check_output(["git", "-C", str(CHECKOUT), "rev-parse", "HEAD"], text=True).strip()
print(f"Installed Representax {REVISION}")

In [ ]:
import jax

print(f"JAX {jax.__version__}")
print(f"backend={jax.default_backend()} processes={jax.process_count()} devices={len(jax.devices())}")
for device in jax.devices():
    print(device)
assert jax.default_backend() == "tpu", "Select the TPU v2 runtime and reconnect"
assert jax.process_count() == 1, "The free-Colab acceptance flow is single-host"
assert len(jax.devices()) >= 2, "Expected a multi-device TPU runtime"

In [ ]:
import datetime

stamp = datetime.datetime.now(datetime.UTC).strftime("%Y%m%dT%H%M%SZ")
OUTPUT = pathlib.Path(f"/content/representax-tpu-acceptance-{stamp}")
device_count = min(8, len(jax.devices()))
subprocess.run(
    [
        sys.executable,
        "-m",
        "experiments.preflights.tpu",
        "--output",
        str(OUTPUT),
        "--steps",
        "20",
        "--device-count",
        str(device_count),
    ],
    cwd=CHECKOUT,
    check=True,
)

In [ ]:
import json
import pandas as pd

summary = json.loads((OUTPUT / "summary.json").read_text())
display(pd.DataFrame(summary["variants"].values()).set_index("name"))
display(pd.DataFrame(summary["parity"]).T)
assert summary["accepted"] is True
print("Representax TPU acceptance: PASS")


In [ ]:
import shutil
from google.colab import files
from IPython.display import FileLink, display

archive = shutil.make_archive(str(OUTPUT), "gztar", root_dir=OUTPUT)
display(FileLink(archive))
files.download(archive)